---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Introduction to Selenium

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)


# 🌙 1. About Selenium

## What's Selenium?
Selenium is a software testing framework used for automating web browsers (such as Chrome, Firefox, Safari, etc.).

Selenium is widely used for application testing and data scraping. We can ask it to open a page, click a button, type in a box, and read what is on the page. In other words, we are controlling a browser through Python!

## How does it work?

1. **WebDriver:** This is the interface we use to control the browser. The browser also needs a small program that carries out these instructions:
   - ChromeDriver for Google Chrome
   - GeckoDriver for Mozilla Firefox
   - EdgeDriver for Microsoft Edge
   - ... and so on.
2. **Coding:** We write Python code to tell the browser what to do.
3. **Execution:** Selenium sends our instructions to the browser's driver, which carries them out in the browser.

Remember Beautiful Soup? It helped us read HTML. Selenium can also **interact** with a webpage -- e.g., click a button that loads more reviews. We still need to figure out which button we want to click!

We'll start by opening a browser. Then we'll find things on a page, fill out a form, log in and out of a demo website, and navigate between pages. Run the cells from the top, in order.


---
# 👩🏻‍💻 2. Installing Selenium

1. Install [Google Chrome](https://www.google.com/chrome/) if you don't already have it. We'll use Chrome for the examples below.
2. Selenium is already listed as a course dependency. Open a terminal in the `wa` course folder and run `uv sync`, then select the Python interpreter whose path contains `.venv` as your notebook kernel.

Do not install Selenium from inside the notebook.

Selenium also needs ChromeDriver to talk to Chrome. The good news: [Selenium Manager](https://www.selenium.dev/documentation/selenium_manager/) takes care of finding and downloading it for us. Keep your internet connection on, and give the first browser-opening cell a little time to finish. You don't need to download a driver manually.

**If you have any problem during the installation, please email the TA**, including the full error message and a screenshot of the whole window.


#### Importing our tools

We'll use four things below. Don't worry about memorizing these long import statements -- the comments tell us what each tool does.


In [ ]:
# webdriver lets Python open and control a browser
from selenium import webdriver

# By tells Selenium HOW to look for an element: by id, name, CSS selector, etc.
from selenium.webdriver.common.by import By

# WebDriverWait lets us wait for something specific to happen on a page
from selenium.webdriver.support.ui import WebDriverWait

# expected_conditions contains the things we can wait for
# EC is just a shorter name, like np for numpy
from selenium.webdriver.support import expected_conditions as EC


#### Opening a browser

`browser` is a Python object representing the browser session. Its methods let us give instructions to Chrome; it is not the webpage itself.


In [ ]:
# create a browser object -- this should open a new Chrome window
browser = webdriver.Chrome()

# give a page navigation at most 30 seconds to finish
browser.set_page_load_timeout(30)

# wait up to 20 seconds for a particular element or condition
# this doesn't make every command take 20 seconds!
wait = WebDriverWait(browser, 20)


If you've done everything correctly, a new Chrome window should open!
- You can use this browser regularly, but you can also control it *through the Python code you're writing here!*
- This is a separate browser window, not your regular Chrome session. It won't automatically have your saved logins.
- Keep it open while working through the next few sections.

If you would like to use a different browser (not Chrome), the commands below are alternatives. Don't run all of them -- we are using Chrome here. Other browsers have their own setup; for example, Safari requires "Allow Remote Automation" in its Develop menu.


In [ ]:
# browser = webdriver.Firefox()
# browser = webdriver.Safari()
# browser = webdriver.Edge()


---
# 🌐 3. Visiting a website

Now, having our browser open through Selenium, let's visit the most boring website possible.


In [ ]:
# get() opens the URL in our Selenium-controlled browser
browser.get("https://www.apostolos-filippas.com/")

# title is the text in the browser tab; current_url is the address we're at
print(browser.title)
print(browser.current_url)


You can also check out all the other methods and attributes using `dir()`. There are a lot of them! We don't need to learn them all today.


In [ ]:
# names starting with '_' are internal details -- let's hide those
browser_tools = [name for name in dir(browser) if not name.startswith('_')]
# only print a short preview; browser_tools still holds the full public list
print(browser_tools[:15])


---
# 🔎 4. How do we locate things on a webpage?

Looking at a page, we can easily see a text box or a button. Python cannot just guess which one we mean! We need to describe the element using its HTML.

Let's use a small [practice form from Selenium](https://www.selenium.dev/selenium/web/web-form.html). This is a demo page, not a real account-registration form.


In [ ]:
browser.get("https://www.selenium.dev/selenium/web/web-form.html")

# get() finishing doesn't always mean every element is ready
# wait until the text box with this id is visible
text_box = wait.until(
    EC.visibility_of_element_located((By.ID, 'my-text-id'))
)
print(browser.title)


#### First, inspect the element

Do this in the Chrome window that Selenium opened:

1. Right-click the **Text input** box and choose **Inspect**.
2. Chrome opens Developer Tools. In the **Elements** tab, it highlights the HTML for what you clicked. If it highlighted a label instead, select the nearby `<input>`.
3. Look for the tag and its attributes. For our text box, you'll find something like this:

```html
<input type="text" class="form-control" name="my-text" id="my-text-id">
```

What does this mean?
- `input` is the **tag**: this is an input field.
- `type="text"` tells us what kind of input it is.
- `id="my-text-id"` identifies this particular element. An id should be unique on the page.
- `name="my-text"` is the field's name.
- `class="form-control"` is a class used for styling. Many elements can share the same class!

These are HTML attributes, not Python variables. Also, use **Inspect**, not just "View page source": JavaScript may have changed the page since it first loaded. The Elements tab shows the current page structure, also called the **DOM**.


#### Turn the HTML into a locator

A **locator** tells Selenium how to find something. We provide two things: **how to search**, and **what value to look for**.

For our text box, these all describe the same element:

| Search by | What we write | What it means |
| --- | --- | --- |
| id | `By.ID, 'my-text-id'` | An element with this exact id |
| name | `By.NAME, 'my-text'` | An element with this exact name |
| CSS selector | `By.CSS_SELECTOR, "input[name='my-text']"` | An input whose name is my-text |
| XPath | `By.XPATH, "//input[@name='my-text']"` | An input anywhere in the document with this name |

CSS selectors are the same kind of selectors we used with Beautiful Soup. A few useful patterns:
- `#my-text-id`: an id (the `#` means id).
- `.form-control`: a class (the dot means class).
- `input[name='my-text']`: a tag with a particular attribute.
- `button[type='submit']`: a button whose type is submit.
- `form input`: inputs **inside** a form (the space means a descendant).

Prefer something short and meaningful, such as a stable id or name. A long "Copy XPath" path based on the element's position may break if the website adds another element.

To test a CSS selector, click inside Chrome's **Elements** tab and press **Command + F** (Mac) or **Ctrl + F** (Windows). Paste `input[name='my-text']`. Check both what it highlights and how many matches it finds. For this box, we want one match, not ten! [More on locators](https://www.selenium.dev/documentation/webdriver/elements/locators/).


In [ ]:
# find_element() returns the FIRST matching WebElement
# a WebElement is a Python object representing one element on the page
text_box = browser.find_element(By.ID, 'my-text-id')

# all three locators below should find that same text box
by_name = browser.find_element(By.NAME, 'my-text')
by_css = browser.find_element(By.CSS_SELECTOR, "input[name='my-text']")
by_xpath = browser.find_element(By.XPATH, "//input[@name='my-text']")

print(text_box == by_name == by_css == by_xpath)  # should be True

# let's look at the element we found, rather than just assuming it's correct
print(text_box.tag_name)                    # input
print(text_box.get_attribute('name'))       # my-text
print(text_box.get_attribute('outerHTML'))  # the HTML for this one element


#### One element vs. many elements

Notice the difference between `find_element()` and `find_elements()`:
- `find_element()` returns the **first** match. If it finds nothing, it raises a `NoSuchElementException`.
- `find_elements()` returns a **list** of all matches. If it finds nothing, it returns an empty list.

Both search the page as it is now. They don't automatically wait for a JavaScript-loaded element to appear.

A broad selector can work perfectly and still find the wrong thing! For example, this form has several inputs with class `form-control`. [More on finding elements](https://www.selenium.dev/documentation/webdriver/elements/finders/).


In [ ]:
# a class is often shared by MANY elements
# By.CLASS_NAME takes ONE class name, without a dot or spaces
fields = browser.find_elements(By.CLASS_NAME, 'form-control')
print(f'We found {len(fields)} elements with this class')

# inspect a small preview of our list
for field in fields[:3]:
    print(field.tag_name, field.get_attribute('name'))

# this made-up id does not exist, so find_elements() returns []
missing_elements = browser.find_elements(By.ID, 'this-id-does-not-exist')
print(missing_elements)


#### Waiting for the right thing

Opening a webpage and finding something on it are not always the same step. Some websites load their buttons and forms **after** the initial page has loaded.

That's why we use `WebDriverWait`, rather than keep telling Python to sleep for five seconds:
- `visibility_of_element_located`: wait for an element to exist and be visible.
- `element_to_be_clickable`: wait for it to be visible and enabled. An overlay can still get in the way!
- `url_contains`: wait until the current URL contains some text.

In `wait.until(EC.visibility_of_element_located((By.ID, 'my-text-id')))`, the inner pair `(By.ID, 'my-text-id')` is our locator. `EC` describes the condition; `until()` keeps checking it. When it succeeds, we get the element back and can use it.

The 30-second page-load timeout and the 20-second element wait do different jobs. Neither is a mandatory pause. If our condition doesn't happen within 20 seconds, we'll get a `TimeoutException`. [More on waits](https://www.selenium.dev/documentation/webdriver/waits/).


---
# ⌨️ 5. Typing and clicking

Finding an element is only the first step. Now we can do something with it!

For an input box:
- `clear()` removes what is already in it.
- `send_keys()` types into it.
- `.text` is for visible text, such as a link or heading. To read what is **inside an input**, use its `value` instead.


In [ ]:
# find the box again, clear it, and type our text
text_box = browser.find_element(By.ID, 'my-text-id')
text_box.clear()
text_box.send_keys('Hello from Python!')
print(text_box.get_attribute('value'))

# CSS selector: a button whose type attribute is 'submit'
submit_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']"))
)
print(submit_button.text)  # check that we found the Submit button


Let's click the button. This submits only the practice form. We'll wait for the confirmation rather than assume the click worked.

Once we move to a new page, don't keep using an element from the old one -- it may no longer exist. Find the new element on the new page.


In [ ]:
# click() works like clicking the button with your mouse
submit_button.click()

# find the confirmation on the NEW page
message = wait.until(EC.visibility_of_element_located((By.ID, 'message')))
print(browser.current_url)
print(message.text)  # should say: Received!


Now close this browser with `quit()`. This closes all its windows and ends the Selenium session. `close()` only closes the current window.

If you want to repeat the form example, run the browser-opening cell again first. The examples below create their own fresh browser sessions.


In [ ]:
# close the browser and end this session
browser.quit()


---
# 🔐 6. Logging in and out of a website

The following code will allow us to log in to a website with Selenium, and then log out!

We'll use [OrangeHRM's public demo website](https://opensource-demo.orangehrmlive.com/). Its demo username is `Admin` and its demo password is `admin123`. These are public demo credentials -- **don't put your own passwords in a notebook you plan to share!**

Before running the code, you can open the page in Chrome and inspect the two boxes and the Login button:
- The username input has `name="username"`, so we use `By.NAME, 'username'`.
- The password input has `name="password"`.
- The Login button is a `button` with `type="submit"`, so we use `button[type='submit']`.

After logging in, inspect the user menu. Its class includes `oxd-userdropdown-tab`. In CSS, `.oxd-userdropdown-tab` means "an element with this class." The Logout option is a link, so `By.LINK_TEXT, 'Logout'` looks for a link with that exact visible text. These locators came from inspecting the page, not from guessing!


#### Let's put the steps together

We've given the boxes and buttons their own variable names so we can see what each step does.

`try` contains the steps we want to run. `finally` closes this example's browser even if one of those steps raises an error; it **doesn't hide the error**. We can then fix the problem and rerun this whole cell.


In [ ]:
# a new session for our login example
browser = webdriver.Chrome()
browser.set_page_load_timeout(30)
wait = WebDriverWait(browser, 20)

try:
    browser.get("https://opensource-demo.orangehrmlive.com/")

    # this site builds its form with JavaScript -- wait for the boxes
    username_box = wait.until(
        EC.visibility_of_element_located((By.NAME, 'username'))
    )
    password_box = wait.until(
        EC.visibility_of_element_located((By.NAME, 'password'))
    )

    # clear any existing text, then type the PUBLIC demo credentials
    username_box.clear()
    username_box.send_keys('Admin')
    password_box.clear()
    password_box.send_keys('admin123')

    # find the Login button by its HTML attributes, then click it
    login_button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']"))
    )
    login_button.click()

    # wait for the post-login URL instead of assuming login was immediate
    wait.until(EC.url_contains('/dashboard'))
    print('Logged in!')

    # the dropdown is only available after login -- locate it on the new page
    user_menu = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, '.oxd-userdropdown-tab'))
    )
    user_menu.click()

    # opening the menu reveals a link with visible text 'Logout'
    logout_link = wait.until(
        EC.element_to_be_clickable((By.LINK_TEXT, 'Logout'))
    )
    logout_link.click()

    # the username box appearing again tells us we're back at the login form
    wait.until(EC.visibility_of_element_located((By.NAME, 'username')))
    print('Logged out!')
finally:
    # quit ends the session, even if something above went wrong
    browser.quit()


---
# 🧭 7. Navigating around a website

The following code will allow us to navigate around a website with Selenium. We'll visit Selenium's documentation, click **Downloads**, and then click **Projects**.

Inspect the Downloads link. It is an `<a>` element, and its `href` attribute is its destination. `By.LINK_TEXT` uses the visible text "Downloads", not that destination.

Before clicking, let's look at the element:
- `.tag_name`: the HTML tag, e.g., `a` for a link.
- `.text`: its visible text.
- `.location`: its x/y position on the page. Useful to inspect, but we don't need to hard-code coordinates!
- `.get_attribute('href')`: the link's destination.

After we click Downloads, we find Projects **again on the new page**. We don't reuse an element object from a page we've left.


In [ ]:
browser = webdriver.Chrome()
browser.set_page_load_timeout(30)
wait = WebDriverWait(browser, 20)

try:
    browser.get("https://www.selenium.dev/documentation/")

    # LINK_TEXT searches links by their exact visible text
    element = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Downloads')))

    # let's take a look at our link before we click it
    print(element.tag_name)
    print(element.text)
    print(element.location)
    print(element.get_attribute('href'))

    element.click()
    wait.until(EC.url_contains('/downloads'))
    print(browser.current_url)

    # we're on another page now, so find its Projects link
    projects_link = wait.until(
        EC.element_to_be_clickable((By.LINK_TEXT, 'Projects'))
    )
    projects_link.click()
    wait.until(EC.url_contains('/projects'))
    print(browser.current_url)

    # we can also navigate using the browser's history
    browser.back()
    wait.until(EC.url_contains('/downloads'))
    print('Back:', browser.current_url)

    browser.forward()
    wait.until(EC.url_contains('/projects'))
    print('Forward:', browser.current_url)
finally:
    browser.quit()


---
# 🛠️ 8. If something doesn't work

First, look at the browser window and the full error message. What page are we actually on? Is the box or button we want visible?

- **Driver-version or browser-opening error:** Update Chrome, run `uv sync` in the course folder, and restart the notebook kernel. Selenium Manager needs internet access to fetch the matching driver.
- **NoSuchElementException:** Our locator didn't find anything in the current page. Check the HTML, spelling, and whether we're on the right page.
- **TimeoutException:** The condition we were waiting for didn't happen in time. Check whether the page loaded correctly, whether the locator still matches, and whether login failed. Don't just keep increasing the timeout!
- **ElementClickInterceptedException:** Something, such as a popup, may cover the button. Inspect the page and dismiss it through the site's normal controls before trying again.
- **StaleElementReferenceException:** We saved an element, but the page changed or we navigated away. Find the element again on the current page.
- **InvalidSessionIdException:** We may have already closed this browser with `quit()`. Start a new session; don't keep using the closed one.

For the first browser session, run its `browser.quit()` cell when you're done, even if an earlier cell failed. The login and navigation examples do this automatically with `finally`.

If the page looks different, inspect it again and update the locator. Public demo sites can change, too. If a website presents a CAPTCHA or browser check, don't try to bypass it; use an allowed teaching example instead.

**If you still cannot solve the problem, please email the TA**, including the full error message and a screenshot of the whole window.

That's it! We can now open a browser, locate elements, type, click, and move between pages. Next, we'll use these tools in `06.3 selenium-booking(class).ipynb` to collect data.
